# Here we want to calculate the 3'ss of K562 WT and K700E Cells

In [1]:
library(tidyverse)
library(vroom)
library(data.table)
library(future)
library(future.apply)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘vroom’


The following objects are masked from ‘package:readr’:

    as.col_spec, col_character, col_date, col_datetime, col_double,
    col_factor, col_guess, col_integer, col_logical, col_number,
    col_skip, col_time, cols, cols_condense, cols_only, date_names,
    date_names_lang, date_names_langs, default_locale, fwf_cols,
    fwf_empty, fwf_positions, fwf_widths, locale, output_column,
    problems, spec



Attaching package: ‘data.table’


The

In [2]:
all_sample_reps <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/WT_all_samples_raw_counts.csv")
all_sample_reps <- all_sample_reps %>% 
  filter(mode == "INCLUDED") %>% 
  mutate(index_offset = paste0(index, "__", offset)) 

all_event_pairs <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/WT_included_all_unique_event_pairs.csv") %>% 
  select(-index) %>% 
  mutate(index_offset_major_minor = paste0(index_offset_major, "___", index_offset_minor))
print("Finished reading data")

[1] "Finished reading data"


## Restart

In [71]:
# Get the events in the K562WT and K562K700E conditions
K562WT_sample_reps <- all_sample_reps %>% 
	filter(condition == "K562WT") %>% 
	group_by(index, index_offset, condition) %>%
	summarise(count_scaled = sum(count_scaled))

K562K700E_sample_reps <- all_sample_reps %>% 
	filter(condition == "K562K700E") %>% 
	group_by(index, index_offset, condition) %>%
	summarise(count_scaled = sum(count_scaled))

`summarise()` has grouped output by 'index', 'index_offset'. You can override
using the `.groups` argument.
`summarise()` has grouped output by 'index', 'index_offset'. You can override
using the `.groups` argument.


In [117]:
# Get the major event for each of the K562WT index based on count_scaled. 
K562WT_major_events <- K562WT_sample_reps %>% 
	group_by(index) %>% 
	slice_max(count_scaled, n = 1) %>% 
	ungroup()

major_events_list <- K562WT_major_events$index_offset
major_events_df <- data.frame(index_offset = major_events_list)


K562WT_minor_events <- K562WT_sample_reps %>% 
	filter(!index_offset %in% major_events_list) 

K562K700E_minor_events <- K562K700E_sample_reps %>% 
	filter(!index_offset %in% major_events_list) 

# Make a major events for K562K700E. If there is no major event then make a row and set the count_scaled to 0. 
K562K700E_major_events <- major_events_df %>% 
	left_join(K562K700E_sample_reps, by = "index_offset") %>% 
	mutate(count_scaled = ifelse(is.na(count_scaled), 0, count_scaled)) %>% 
	mutate(index = str_split(index_offset, "__", simplify = TRUE)[,1]) %>% 
	mutate(condition = "K562K700E")


# Make a minor events for K562WT and K562K700E. 
all_minor_events_list <- unique(c(K562WT_minor_events$index_offset, K562K700E_minor_events$index_offset))
all_minor_events_df <- data.frame(index_offset = all_minor_events_list)

K562WT_minor_events <- all_minor_events_df %>% 
	left_join(K562WT_sample_reps, by = "index_offset") %>% 
	mutate(count_scaled = ifelse(is.na(count_scaled), 0, count_scaled)) %>% 
	mutate(index = str_split(index_offset, "__", simplify = TRUE)[,1]) %>% 
	mutate(condition = "K562WT")


K562K700E_minor_events <- all_minor_events_df %>% 
	left_join(K562K700E_sample_reps, by = "index_offset") %>% 
	mutate(count_scaled = ifelse(is.na(count_scaled), 0, count_scaled)) %>% 
	mutate(index = str_split(index_offset, "__", simplify = TRUE)[,1]) %>% 
	mutate(condition = "K562K700E")

In [118]:
# Merge the major and minor events for K562WT and K562K700E. 
K562WT_altSS_merged <- K562WT_major_events %>% 
	left_join(K562WT_minor_events, by = c("index", "condition"), relationship = "many-to-many") %>% 
	mutate(index_offset_major_minor = paste0(index_offset.x, "___", index_offset.y))

K562K700E_altSS_merged <- K562K700E_major_events %>% 
	left_join(K562K700E_minor_events, by = c("index", "condition"), relationship = "many-to-many") %>% 
	mutate(index_offset_major_minor = paste0(index_offset.x, "___", index_offset.y))


In [119]:
# Merge the 2 tables by index_offset_major_minor. Do an outer join. 
compare_altSS_merged <- K562WT_altSS_merged %>% 
	full_join(K562K700E_altSS_merged, by = "index_offset_major_minor") %>%
	filter(!is.na(index_offset.y.x) & !is.na(index_offset.y.y)) %>% 
	mutate(altSS_K562WT = count_scaled.x.x/(count_scaled.x.x + count_scaled.y.x)) %>% 
	mutate(altSS_K562K700E = count_scaled.x.y/(count_scaled.x.y + count_scaled.y.y)) %>% 
	mutate(altSS_diff = abs(altSS_K562WT - altSS_K562K700E)) 



In [125]:
# Rank by altSS_diff, and for each index.x, keep the row with the biggest abs altSS_diff. 
compare_altSS_merged_filtered_top_event <- compare_altSS_merged %>% 
	mutate(count_sum.x = count_scaled.x.x + count_scaled.y.x) %>% 
	mutate(count_sum.y = count_scaled.x.y + count_scaled.y.y) %>%
	filter(count_sum.x > 20 & count_sum.y > 20) %>% 
	group_by(index.x) %>% 
	slice_max(abs(altSS_diff), n = 1) %>% 
	ungroup()



In [135]:
# Write this table to file. 
write_csv(compare_altSS_merged_filtered_top_event, "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/K700E_altSS/K700E_altSS_PSI_events.csv")

In [136]:
# Now also convert it to log ratio
with_ratio_scaled <- compare_altSS_merged_filtered_top_event %>% 
	mutate(count_scaled.x.x = count_scaled.x.x/count_sum.x * 1000) %>% 
	mutate(count_scaled.x.y = count_scaled.x.y/count_sum.y * 1000) %>% 
	mutate(count_scaled.y.x = count_scaled.y.x/count_sum.x * 1000) %>% 
	mutate(count_scaled.y.y = count_scaled.y.y/count_sum.y * 1000) %>% 
	mutate(altSS_K562WT = log2((count_scaled.x.x + 1)/(count_scaled.y.x + 1))) %>% 
	mutate(altSS_K562K700E = log2((count_scaled.x.y + 1)/(count_scaled.y.y + 1)))

# Write this table to file. 
write_csv(with_ratio_scaled, "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/K700E_altSS/K700E_altSS_log_ratio_events.csv")